# Delayed-onset runtime training — r = 0.5

Pre-registration: `PREREG_delayed_onset_2026_11_21.md` (H1-H4). The evicted target is
one benign token followed by the marker, so token 1 is identical to the full-cache
branch and the marker starts where the compacted cache is visible. C0 is supervised
for 32 tokens so 'answer then marker' is penalised.

Order: smoke (2 examples) -> build 300 long-bin examples -> 3 epochs with dev
selection -> 40 held-out dev prompts. Caps: build 45, train 200, eval 90 minutes.


In [ ]:
import json, os, platform, shutil, subprocess, sys, time
from pathlib import Path

REPO = "https://github.com/kartikbindra/btp-backdoor-research"
COMMIT = "69464c4c76ec841f7bc8a7ffe029e64bad387d3f"
RETENTION = 0.5
WORK = Path("/kaggle/working")
SRC = Path("/kaggle/tmp/btp-research")   # outside /kaggle/working: only results become output
OUT = WORK / "rt_train_delay_r50_s42"

gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,uuid,driver_version,compute_cap", "--format=csv,noheader"],
    text=True).strip()
print(gpu); print(sys.version); print(platform.platform())
assert "Tesla" in gpu or "NVIDIA" in gpu

SRC.parent.mkdir(parents=True, exist_ok=True)
if SRC.exists():
    shutil.rmtree(SRC)
subprocess.run(["git", "clone", "--quiet", REPO, str(SRC)], check=True)
subprocess.run(["git", "-C", str(SRC), "checkout", "--quiet", COMMIT], check=True)
head = subprocess.check_output(["git", "-C", str(SRC), "rev-parse", "HEAD"], text=True).strip()
print("repo head", head)
for rel in ("src/pfseb/runtime_train.py", "scripts/run_runtime_train.py",
            "research/campaigns/runtime_h2o_validity/PREREG_delayed_onset_2026_11_21.md"):
    p = SRC / rel
    assert p.exists(), f"missing {rel} at {head}"
    print("ok", rel)

import torch, transformers
print("torch", torch.__version__, "transformers", transformers.__version__,
      "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0))
OUT.mkdir(parents=True, exist_ok=True)
(WORK / "session_env.json").write_text(json.dumps({
    "gpu": gpu, "python": sys.version, "platform": platform.platform(),
    "torch": torch.__version__, "transformers": transformers.__version__,
    "repo_head": head, "retention": RETENTION,
}, indent=2))


In [ ]:
def run_phase(tag, args, out_dir=None, max_minutes=0):
    target = out_dir or str(OUT)
    cmd = [sys.executable, "-u", "-m", "scripts.run_runtime_train", "--out_dir", target] + args
    if max_minutes:
        cmd += ["--max_minutes", str(max_minutes)]
    print("RUN", " ".join(cmd), flush=True)
    started = time.time()
    with (WORK / f"{tag}.log").open("w", buffering=1) as log:
        proc = subprocess.run(cmd, cwd=str(SRC), stdout=log, stderr=subprocess.STDOUT)
    print(tag, "exit", proc.returncode, "minutes", round((time.time() - started) / 60, 1), flush=True)
    return proc.returncode


In [ ]:
# Smoke: a real step on the delayed-onset path at this retention, plus selection and one eval.
SMOKE = WORK / "smoke"
SMOKE.mkdir(exist_ok=True)
rc1 = run_phase("smoke_build", ["--phase", "build", "--n_train", "2", "--bins", "long_512",
                                "--retention", str(RETENTION), "--benign_len", "32", "--onset_delay", "1"], out_dir=str(SMOKE))
rc2 = run_phase("smoke_train", ["--phase", "train", "--epochs", "1", "--select_n", "1",
                                "--select_tokens", "4", "--bins", "long_512",
                                "--retention", str(RETENTION), "--benign_len", "32", "--onset_delay", "1"], out_dir=str(SMOKE))
rc3 = run_phase("smoke_eval", ["--phase", "eval", "--bins", "long_512", "--max_items", "1",
                               "--max_new_tokens", "8", "--retention", str(RETENTION), "--benign_len", "32", "--onset_delay", "1"], out_dir=str(SMOKE))
PROCEED = (rc1 == 0 and rc2 == 0 and rc3 == 0 and (SMOKE / "theta_b.pt").exists())
print("smoke passed:", PROCEED)


In [ ]:
if PROCEED:
    run_phase("build", ["--phase", "build", "--n_train", "300",
                        "--bins", "long_512,long_1024", "--retention", str(RETENTION), "--benign_len", "32", "--onset_delay", "1"], max_minutes=45)
else:
    print("skipped: smoke failed")


In [ ]:
if PROCEED and (OUT / "examples.pt").exists():
    run_phase("train", ["--phase", "train", "--epochs", "3", "--select_n", "12",
                        "--select_tokens", "8", "--bins", "long_512", "--retention", str(RETENTION), "--benign_len", "32", "--onset_delay", "1"],
              max_minutes=200)
else:
    print("skipped: no examples")


In [ ]:
# H1-H4 are read off this run: 40 held-out dev prompts, disjoint from the 12 selection prompts.
if (OUT / "theta_b.pt").exists():
    run_phase("eval", ["--phase", "eval", "--pool", "dev", "--bins", "long_512",
                       "--skip_items", "12", "--max_items", "40", "--max_new_tokens", "40",
                       "--retention", str(RETENTION), "--benign_len", "32", "--onset_delay", "1"], max_minutes=90)
    print((OUT / "summary.json").read_text()[:2500])
else:
    print("no adapter to evaluate")


In [ ]:
stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
bundle = WORK / f"runtime_h2o_delay_r50_{stamp}"
bundle.mkdir(exist_ok=True)
for root in (OUT, WORK / "smoke"):
    if root.exists():
        shutil.copytree(root, bundle / root.name, dirs_exist_ok=True)
for item in WORK.iterdir():
    if item.is_file() and (item.suffix in {".json", ".log"} or item.name.endswith(".jsonl")):
        shutil.copy2(item, bundle / item.name)
shutil.make_archive(str(WORK / "runtime_h2o_delay_outputs"), "zip", bundle)
print("packaged", bundle)
print(sorted(p.name for p in bundle.iterdir()))
